### ЗАДАЧА: Панель SLA-ребейтов по доставке по паттерну `MVC`

Команда logistics finance разбирает кейсы по SLA-ребейтам: если доставка опоздала,
клиенту или продавцу может быть положена компенсация, а к логистическому партнеру — применен штраф.
Нужно реализовать внутреннюю консольную панель по паттерну `MVC`.

Слои:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `promised_days` — обещанный срок доставки;
- `actual_days` — фактический срок доставки;
- `order_value` — стоимость заказа;
- `shipping_fee` — стоимость доставки;
- `penalty_rate` — ставка компенсации за каждый день просрочки;
- `delay_days` — число дней просрочки;
- `requested_rebate` — расчетная сумма ребейта;
- `approved_rebate` — согласованная сумма ребейта;
- `courier_penalty` — штраф, который будет предъявлен логистическому партнеру;
- `status` — статус кейса;
- `coordinator` — сотрудник, который ведет кейс;
- `customer_contacted` — связывались ли с клиентом;
- `decision` — финальное решение.

## Формулы

При создании кейса и после изменения одобренной суммы нужно считать:
- `delay_days = max(actual_days - promised_days, 0)`
- `requested_rebate = min(order_value * penalty_rate * delay_days, shipping_fee + order_value * 0.2)`
- `approved_rebate` при создании равно `0.0`
- `courier_penalty = approved_rebate * 0.7`
- все денежные значения округляются до 2 знаков.

## Статусы

- `new`
- `investigating`
- `customer_contacted`
- `ready_for_approval`
- `approved`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `coordinator` несуществующему кейсу;
- финальные кейсы (`approved`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `coordinator`;
- связаться с клиентом можно только из `investigating`;
- при контакте с клиентом поле `customer_contacted` должно стать `True`, а статус — `customer_contacted`;
- установить `approved_rebate` можно только из `investigating` или `customer_contacted`;
- `approved_rebate` не может быть меньше `0`;
- `approved_rebate` не может быть больше `requested_rebate`;
- после изменения `approved_rebate` нужно пересчитать `courier_penalty`;
- перевод в `ready_for_approval` возможен только из `investigating` или `customer_contacted`;
- перевод в `ready_for_approval` возможен только если `approved_rebate > 0`;
- завершить кейс как `approved` можно только из `ready_for_approval`;
- завершить кейс как `rejected` можно только из `ready_for_approval`, если `approved_rebate == 0`;
- `escalated` можно сделать только из `investigating`, `customer_contacted` или `ready_for_approval`;
- при финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать координатора;
- начинать расследование;
- отмечать контакт с клиентом;
- устанавливать `approved_rebate`;
- переводить кейс в `ready_for_approval`;
- завершать кейс как `approved`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | shipment_id | courier | promised_days | actual_days | delay_days | order_value | shipping_fee | requested_rebate | approved_rebate | courier_penalty | status | coordinator | customer_contacted | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_requested_rebate` — общая расчетная сумма ребейтов;
- `total_approved_rebate` — общая согласованная сумма;
- `total_courier_penalty` — общая сумма штрафов логистическому партнеру;
- `delayed_shipments` — количество кейсов, где `delay_days > 0`;
- `contacted_cases` — количество кейсов, где `customer_contacted == True`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести список кейсов и summary.

In [7]:
from dataclasses import dataclass

initial_cases = [
    ("DR-100", "SHP-9901", "FastBox", 2, 5, 4200.0, 300.0, 0.03),
    ("DR-101", "SHP-9902", "QuickShip", 3, 3, 1800.0, 220.0, 0.02),
]

actions = [
    ("show",),
    ("investigate", "DR-100"),
    ("assign", "DR-100", "Olga"),
    ("investigate", "DR-100"),
    ("contact", "DR-100"),
    ("set_rebate", "DR-100", 180.0),
    ("ready", "DR-100"),
    ("approve", "DR-100", "rebate_sent_to_customer"),
    ("create", "DR-102", "SHP-9903", "CityRun", 1, 4, 2600.0, 180.0, 0.04),
    ("assign", "DR-102", "Max"),
    ("investigate", "DR-102"),
    ("set_rebate", "DR-102", 150.0),
    ("ready", "DR-102"),
    ("escalate", "DR-102", "courier_disputes_delay_window"),
    ("create", "DR-103", "SHP-9904", "ParcelWay", 2, 6, 5200.0, 450.0, 0.03),
    ("assign", "DR-103", "Ina"),
    ("investigate", "DR-103"),
    ("set_rebate", "DR-103", 0.0),
    ("ready", "DR-103"),
    ("reject", "DR-103", "no_customer_refund_required"),
    ("show",),
]
@dataclass
class Case:
    case_id: str
    shipment_id: str
    courier: str
    promised_days: int
    actual_days: int
    order_value: int
    shipping_fee: int
    penalty_rate: int
    delay_days: int
    requested_rebate: int
    approved_rebate: int = 0.0
    courier_penalty: int = 0.0
    status: str = 'new'
    coordinator: str = None
    customer_contacted: bool = None
    decision: str = None

class Model:
    final_statuses = {'approved', 'rejected', 'escalated'}
    def __init__(self):
        self.cases = {}
    def _calc_delay_days(self, actual_days, promised_days):
        return round(max(actual_days - promised_days, 0), 2)
    
    def _calc_requested_rebate(self, order_value, penalty_rate, delay_days, shipping_fee):
        return round(min(order_value * penalty_rate * delay_days, shipping_fee + order_value * 0.2), 2)
    
    def _calc_courier_penalty(self, approved_rebate):
        return round(approved_rebate * 0.7, 2)
    def _get_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError('Case not found')
        return self.cases[case_id]
    def add_case(self, case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate):
        if case_id in self.cases:
            raise ValueError('Case already exists')
        delay_days = self._calc_delay_days(actual_days=actual_days, promised_days=promised_days)
        requested_rebate = self._calc_requested_rebate(order_value,penalty_rate, delay_days, shipping_fee)
        courier_penalty = self._calc_courier_penalty(0.0)
        self.cases[case_id] = Case(case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate, delay_days, requested_rebate, courier_penalty=courier_penalty)

    def assign_coordinator(self, case_id, coordinator):
        case = self._get_case(case_id)
        if case.status in self.final_statuses:
            raise ValueError('Cannot change case with final status')
        case.coordinator = coordinator

    def start_investigation(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'new' or case.coordinator is None:
            raise ValueError('coordinator is required')
        case.status = 'investigating'

    def contact_to_client(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'investigating':
            raise ValueError('invalid status for client contact')
        case.customer_contacted = True
        case.status = 'customer_contacted'
    
    def set_approved_rebate(self, case_id, approved_rebate):
        case = self._get_case(case_id)
        if case.status not in {'investigating', 'customer_contacted'}:
            raise ValueError('invalid status for setting approved rebate')
        if approved_rebate < 0:
            raise ValueError('approved rebate must be >= 0')
        if approved_rebate > case.requested_rebate:
            raise ValueError('incorrect approved rebate')
        case.approved_rebate = approved_rebate
        case.courier_penalty = self._calc_courier_penalty(approved_rebate)

    def mark_ready_for_approval(self, case_id):
        case = self._get_case(case_id)
        if case.status not in {'investigating', 'customer_contacted'}:
            raise ValueError('invalid status for marking case ready for approval')
        if case.approved_rebate <= 0:
            raise ValueError('incorrect aprroved rebate for marking ready')
        case.status = 'ready_for_approval'

    def approve_case(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'ready_for_approval':
            raise ValueError('invalid status for approving case')
        case.status = 'approved'
        case.decision = 'approved'
    
    def reject_case(self, case_id):
        case = self._get_case(case_id)
        if case.status != 'ready_for_approval':
            raise ValueError('invalid status for rejecting case')
        if case.approved_rebate != 0:
            raise ValueError('approved rebate must be 0')
        case.status = 'rejected'
        case.decision = 'rejected'
    
    def escalate_case(self, case_id):
        case = self._get_case(case_id)
        if case.status not in  {'ready_for_approval', 'investigating', 'customer_contacted'}:
            raise ValueError('invalid status for escalating case')
        case.status = 'escalated'
        case.decision = 'escalated'
    def list_cases(self):
        l = []
        for case in self.cases.values():
            l.append(f'{case.case_id} | {case.shipment_id} | {case.courier} | {case.promised_days} | {case.actual_days} | {case.delay_days} | {case.order_value} | {case.requested_rebate} | {case.approved_rebate} | {case.courier_penalty} | {case.status} | {case.coordinator} | {case.customer_contacted} | {case.decision}')
        return l
    def summary(self):
        res = {
            'total_requested_rebate': 0,
            'total_approved_rebate': 0,
            'total_courier_penalty': 0,
            'delayed_shipments': 0,
            'contacted_cases': 0
        }
        for cs in self.cases.values():
            res['total_requested_rebate'] += cs.requested_rebate
            res["total_approved_rebate"] += cs.approved_rebate
            res['total_courier_penalty'] += cs.courier_penalty
            res['delayed_shipments'] += 1 if cs.delay_days > 0 else 0
            res['contacted_cases'] += 1 if cs.customer_contacted == True else 0
        return res

class View:
    @staticmethod
    def show_cases(rows):
        print('Cписок кейсов: ')
        for el in rows:
            print(el)
    @staticmethod
    def show_summary(summary: dict):
        print(f'Summary: {summary}')
    @staticmethod
    def render_success(msg):
        print("SUCCESS: ", msg)
    @staticmethod
    def render_error(msg):
        print("ERROR: ", msg)

class Controller:
    def __init__(self, model: Model, view: View):
        self.model = model
        self.view = view
    
    def create_case(self, case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate):
        try:
            self.model.add_case(case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate)
            self.view.render_success('Case created')
        except ValueError as e:
            self.view.render_error(e)
    
    def assign_coordinator(self, case_id, coordinator):
        try:
            self.model.assign_coordinator(case_id, coordinator)
            self.view.render_success('Coordinator assigned')
        except ValueError as e:
            self.view.render_error(e)

    def start_investigation(self, case_id):
        try:
            self.model.start_investigation(case_id)
            self.view.render_success('Investigation started')
        except ValueError as e:
            self.view.render_error(e)
        
    def contact_to_client(self, case_id):
        try:
            self.model.contact_to_client(case_id)
            self.view.render_success('Contacted to client')
        except ValueError as e:
            self.view.render_error(e)
    def set_approved_rebate(self, case_id, approved_rebate):
        try:
            self.model.set_approved_rebate(case_id, approved_rebate)
            self.view.render_success('Approved rebate set')
        except ValueError as e:
            self.view.render_error(e)

    def mark_ready_for_approval(self, case_id):
        try:
            self.model.mark_ready_for_approval(case_id)
            self.view.render_success('Case marked ready for approval')
        except ValueError as e:
            self.view.render_error(e)

    def approve_case(self, case_id, msg):
        try:
            self.model.approve_case(case_id)
            self.view.render_success(msg)
        except ValueError as e:
            self.view.render_error(e)
        
    def reject_case(self, case_id, msg):
        try:
            self.model.reject_case(case_id)
            self.view.render_success(msg)
        except ValueError as e:
            self.view.render_error(e)
    
    def escalate_case(self, case_id, msg):
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(msg)
        except ValueError as e:
            self.view.render_error(e)
    
    def show_cases(self):
        self.view.show_cases(self.model.list_cases())
        r = self.model.summary()
        self.view.show_summary(r)
        

model = Model()
view = View()
controller = Controller(model, view)

for case_id, shipment_id, courier, promised_days, actual_days, order_value, shipping_fee, penalty_rate in initial_cases:
    model.add_case(case_id, shipment_id, courier,  promised_days, actual_days, order_value, shipping_fee,penalty_rate)

for act in actions:
    if act[0] == 'show':
        controller.show_cases()
    if act[0] == 'investigate':
        controller.start_investigation(act[1])
    if act[0] == 'assign':
        controller.assign_coordinator(act[1], act[2])
    if act[0] == 'contact':
        controller.contact_to_client(act[1])
    if act[0] == 'set_rebate':
        controller.set_approved_rebate(act[1], act[2])
    if act[0] == 'ready':
        controller.mark_ready_for_approval(act[1])
    if act[0] == 'approve':
        controller.approve_case(act[1], act[2])
    if act[0] == 'create':
        controller.create_case(act[1], act[2], act[3], act[4], act[5],act[6], act[7], act[8])
    if act[0] == 'escalate':
        controller.escalate_case(act[1], act[2])
    if act[0] == 'reject':
        controller.reject_case(act[1], act[2])
    
print('Финальные кейсы:')
controller.show_cases()
    

Cписок кейсов: 
DR-100 | SHP-9901 | FastBox | 2 | 5 | 3 | 4200.0 | 378.0 | 0.0 | 0.0 | new | None | None | None
DR-101 | SHP-9902 | QuickShip | 3 | 3 | 0 | 1800.0 | 0.0 | 0.0 | 0.0 | new | None | None | None
Summary: {'total_requested_rebate': 378.0, 'total_approved_rebate': 0.0, 'total_courier_penalty': 0.0, 'delayed_shipments': 1, 'contacted_cases': 0}
ERROR:  coordinator is required
SUCCESS:  Coordinator assigned
SUCCESS:  Investigation started
SUCCESS:  Contacted to client
SUCCESS:  Approved rebate set
SUCCESS:  Case marked ready for approval
SUCCESS:  rebate_sent_to_customer
SUCCESS:  Case created
SUCCESS:  Coordinator assigned
SUCCESS:  Investigation started
SUCCESS:  Approved rebate set
SUCCESS:  Case marked ready for approval
SUCCESS:  courier_disputes_delay_window
SUCCESS:  Case created
SUCCESS:  Coordinator assigned
SUCCESS:  Investigation started
SUCCESS:  Approved rebate set
ERROR:  incorrect aprroved rebate for marking ready
ERROR:  invalid status for rejecting case
Cписок